# 📡 Daily NSE Signal Scanner
Scans Nifty 100 (or 50/200) every day with the EMA + RSI + ATR strategy and tells you what to **BUY**, **SELL** or **HOLD**.

**Daily use:** run after **3:30 PM IST** → `Runtime → Run all` (takes ~1–2 min).

> ⚠️ Educational tool, not financial advice. Paper trade first: the log in step 7 is for that.

## 1. Install

In [ ]:
!pip install yfinance -q

## 2. Settings

In [ ]:
# ======================= SETTINGS (edit these) =======================
UNIVERSE      = "NIFTY100"   # "NIFTY50", "NIFTY100" or "NIFTY200"
HISTORY_START = "2015-01-01" # long history = more trades = fairer stats
SAVE_TO_DRIVE = False        # True = keep daily results + log in Google Drive

# "Strategy works on this stock" filter (based on past backtest)
MIN_SHARPE    = 0.5
MIN_TRADES    = 10

CONFIG = {
    "capital": 100_000,        # INR per stock (for position sizing)
    "risk_per_trade": 0.01,    # risk 1% of capital per trade
    "ema_fast": 20,
    "ema_slow": 50,
    "rsi_period": 14,
    "rsi_max_entry": 70,
    "atr_period": 14,
    "atr_stop_mult": 2.0,
    "cost_pct": 0.0012,        # ~0.12% per side
}

## 3. Strategy engine

In [ ]:
import numpy as np, pandas as pd, datetime as dt, os, io

def add_indicators(df, c):
    df = df.copy()
    df["ema_fast"] = df["Close"].ewm(span=c["ema_fast"], adjust=False).mean()
    df["ema_slow"] = df["Close"].ewm(span=c["ema_slow"], adjust=False).mean()
    delta = df["Close"].diff()
    gain = delta.clip(lower=0).ewm(alpha=1 / c["rsi_period"], adjust=False).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1 / c["rsi_period"], adjust=False).mean()
    df["rsi"] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))
    pc = df["Close"].shift()
    tr = pd.concat([df["High"] - df["Low"], (df["High"] - pc).abs(), (df["Low"] - pc).abs()], axis=1).max(axis=1)
    df["atr"] = tr.ewm(alpha=1 / c["atr_period"], adjust=False).mean()
    return df.dropna()

def generate_signals(df, c):
    up = (df["ema_fast"] > df["ema_slow"]) & (df["ema_fast"].shift() <= df["ema_slow"].shift())
    dn = (df["ema_fast"] < df["ema_slow"]) & (df["ema_fast"].shift() >= df["ema_slow"].shift())
    df["buy"] = up & (df["rsi"] < c["rsi_max_entry"])
    df["sell"] = dn
    return df

def backtest(df, c):
    """Signal on day t executes at day t+1 open (no look-ahead)."""
    cash, qty, entry_px, stop = c["capital"], 0, 0.0, 0.0
    trades, equity = [], []
    O, H, L, C = df["Open"].values, df["High"].values, df["Low"].values, df["Close"].values
    A, B, S, D = df["atr"].values, df["buy"].values, df["sell"].values, df.index
    for i in range(len(df) - 1):
        j = i + 1
        if qty > 0:
            stop = max(stop, C[i] - c["atr_stop_mult"] * A[i])
            if L[j] <= stop:
                px = min(O[j], stop)
                cash += qty * px * (1 - c["cost_pct"])
                trades.append((D[j], "STOP", px, qty, (px - entry_px) * qty)); qty = 0
        if qty > 0 and S[i]:
            px = O[j]; cash += qty * px * (1 - c["cost_pct"])
            trades.append((D[j], "SELL", px, qty, (px - entry_px) * qty)); qty = 0
        elif qty == 0 and B[i]:
            px, risk = O[j], c["atr_stop_mult"] * A[i]
            size = min(int(cash * c["risk_per_trade"] / risk), int(cash / (px * (1 + c["cost_pct"]))))
            if size > 0:
                qty, entry_px, stop = size, px, px - risk
                cash -= qty * px * (1 + c["cost_pct"])
                trades.append((D[j], "BUY", px, qty, 0.0))
        equity.append((D[j], cash + qty * C[j]))
    eq = pd.Series(dict(equity))
    return eq, pd.DataFrame(trades, columns=["date", "action", "price", "qty", "pnl"])

def stats(eq, tr, df, capital):
    r = eq.pct_change().dropna()
    closed = tr[tr["action"] != "BUY"]
    return {"Strategy %": (eq.iloc[-1] / capital - 1) * 100,
            "Buy&Hold %": (df["Close"].iloc[-1] / df["Close"].iloc[0] - 1) * 100,
            "Sharpe": np.sqrt(252) * r.mean() / r.std() if r.std() > 0 else 0.0,
            "MaxDD %": (eq / eq.cummax() - 1).min() * 100,
            "Trades": len(closed),
            "Win %": (closed["pnl"] > 0).mean() * 100 if len(closed) else 0.0}

def today_signal(df, tr, c):
    last = df.iloc[-1]
    in_pos = len(tr) > 0 and tr.iloc[-1]["action"] == "BUY"
    if in_pos:
        e = tr.iloc[-1]
        since = df.loc[df.index >= e["date"]]
        stop = max(e["price"] - c["atr_stop_mult"] * df.loc[:e["date"]].iloc[-2]["atr"],
                   (since["Close"] - c["atr_stop_mult"] * since["atr"]).max())
        if last["sell"]:
            return "🔴 SELL", round(stop, 2), None, f"held since {e['date'].date()} @ ₹{e['price']:.2f}"
        return "🟡 HOLD", round(stop, 2), None, f"held since {e['date'].date()} @ ₹{e['price']:.2f}"
    if last["buy"]:
        risk = c["atr_stop_mult"] * last["atr"]
        qty = min(int(c["capital"] * c["risk_per_trade"] / risk), int(c["capital"] / last["Close"]))
        return "🟢 BUY", round(last["Close"] - risk, 2), qty, "EMA crossed up today"
    return "⚪ WAIT", None, None, ""

## 4. Stock universe
Pulls the **current official index list** from NSE, so it stays up to date as stocks enter or leave the index.

In [ ]:
import requests

FALLBACK = """ADANIENT ADANIPORTS APOLLOHOSP ASIANPAINT AXISBANK BAJAJ-AUTO BAJFINANCE BAJAJFINSV BEL BHARTIARTL
CIPLA COALINDIA DRREDDY EICHERMOT ETERNAL GRASIM HCLTECH HDFCBANK HDFCLIFE HINDALCO HINDUNILVR ICICIBANK
INDIGO INFY ITC JIOFIN JSWSTEEL KOTAKBANK LT M&M MARUTI MAXHEALTH NESTLEIND NTPC ONGC POWERGRID RELIANCE
SBILIFE SBIN SHRIRAMFIN SUNPHARMA TATACONSUM TATASTEEL TCS TECHM TITAN TRENT ULTRACEMCO WIPRO
DMART HAL DIVISLAB PIDILITIND SIEMENS HAVELLS DABUR GODREJCP BRITANNIA LTIM VEDL TVSMOTOR BANKBARODA PNB
CANBK IOC BPCL GAIL TATAPOWER ADANIGREEN ADANIPOWER AMBUJACEM SHREECEM DLF LODHA ICICIGI ICICIPRULI
CHOLAFIN BAJAJHLDNG PFC RECLTD IRFC ZYDUSLIFE TORNTPHARM MOTHERSON BOSCHLTD NAUKRI INDHOTEL UNITDSPR
VBL JSWENERGY ABB CGPOWER LICI HINDZINC""".split()

def load_universe(name="NIFTY100"):
    """Official current index list from NSE; falls back to a built-in list."""
    f = f"ind_{name.lower()}list.csv"
    urls = [f"https://niftyindices.com/IndexConstituent/{f}",
            f"https://archives.nseindia.com/content/indices/{f}"]
    for url in urls:
        try:
            r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=10)
            r.raise_for_status()
            u = pd.read_csv(io.StringIO(r.text))
            u = u.rename(columns={"Company Name": "Company"})[["Symbol", "Company", "Industry"]]
            print(f"✅ Loaded {len(u)} stocks in {name} from NSE")
            return u
        except Exception:
            continue
    print(f"⚠️ Couldn't reach NSE, using built-in list of {len(FALLBACK)} large caps")
    return pd.DataFrame({"Symbol": FALLBACK, "Company": FALLBACK, "Industry": ""})

universe = load_universe(UNIVERSE)
universe.head()

## 5. Download prices

In [ ]:
import yfinance as yf

def download_all(symbols, start, batch=50):
    data = {}
    tickers = [s + ".NS" for s in symbols]
    for k in range(0, len(tickers), batch):
        chunk = tickers[k:k + batch]
        raw = yf.download(chunk, start=start, group_by="ticker", auto_adjust=True,
                          threads=True, progress=False)
        for t in chunk:
            try:
                d = raw[t][["Open", "High", "Low", "Close", "Volume"]].dropna()
                if len(d) > 200:
                    data[t[:-3]] = d
            except Exception:
                pass
        print(f"Downloaded {min(k + batch, len(tickers))}/{len(tickers)}")
    missing = sorted(set(symbols) - set(data))
    if missing:
        print("⚠️ No data for:", ", ".join(missing))
    return data

prices = download_all(universe["Symbol"].tolist(), HISTORY_START)

## 6. Scan every stock
- **Action**: today's signal (executes at next day's open)
- **Strategy works?** ✅ if this strategy was profitable on this stock historically (Sharpe ≥ 0.5, ≥10 trades)

In [ ]:
rows = []
latest = max(d.index[-1] for d in prices.values())
for sym, raw in prices.items():
    try:
        df = generate_signals(add_indicators(raw, CONFIG), CONFIG)
        eq, tr = backtest(df, CONFIG)
        action, stop, qty, note = today_signal(df, tr, CONFIG)
        st = stats(eq, tr, df, CONFIG["capital"])
        works = st["Sharpe"] >= MIN_SHARPE and st["Trades"] >= MIN_TRADES and st["Strategy %"] > 0
        rows.append({"Symbol": sym, "Action": action, "Close": round(df["Close"].iloc[-1], 2),
                     "Stop-loss": stop, "Qty": qty, "Note": note,
                     "Strategy works?": "✅" if works else "❌",
                     **{k: round(v, 2) for k, v in st.items()},
                     "Data date": df.index[-1].date(),
                     "Stale": df.index[-1] < latest})
    except Exception as e:
        print(sym, "error:", e)

scan = (pd.DataFrame(rows)
        .merge(universe[["Symbol", "Company", "Industry"]], on="Symbol", how="left")
        .set_index("Symbol"))
order = {"🟢 BUY": 0, "🔴 SELL": 1, "🟡 HOLD": 2, "⚪ WAIT": 3}
scan = scan.sort_values(["Action", "Sharpe"], key=lambda s: s.map(order) if s.name == "Action" else -s)

print(f"\nScan for {latest.date()}  |  {len(scan)} stocks")
print(scan["Action"].value_counts().to_string())
stale = scan.index[scan["Stale"]].tolist()
if stale:
    print("⚠️ Older data (check these manually):", ", ".join(stale))

## 7. Today's actions

In [ ]:
cols = ["Company", "Action", "Close", "Stop-loss", "Qty", "Note", "Strategy works?", "Sharpe", "Win %", "Trades"]
for act, title in [("🟢 BUY", "BUY at next open"), ("🔴 SELL", "SELL at next open"), ("🟡 HOLD", "HOLDING (watch stop-loss)")]:
    part = scan[scan["Action"] == act]
    print(f"\n===== {title}: {len(part)} =====")
    if len(part):
        display(part[cols]) if "display" in globals() else print(part[cols].to_string())

## 8. Profitability check on BUY candidates
⭐ = strategy has worked on this stock **and** the company is profitable (profit margin > 0, ROE > 12%).

In [ ]:
# Profitability check for today's BUY candidates (a few API calls only)
def fundamentals(sym):
    try:
        i = yf.Ticker(sym + ".NS").info
        pm, roe = i.get("profitMargins"), i.get("returnOnEquity")
        return {"Profit margin %": round(pm * 100, 1) if pm is not None else None,
                "ROE %": round(roe * 100, 1) if roe is not None else None,
                "Debt/Equity": i.get("debtToEquity"),
                "Profitable?": "✅" if (pm or 0) > 0 and (roe or 0) > 0.12 else "❌"}
    except Exception:
        return {"Profit margin %": None, "ROE %": None, "Debt/Equity": None, "Profitable?": "?"}

buys = scan[scan["Action"] == "🟢 BUY"].index.tolist()
if buys:
    f = pd.DataFrame({s: fundamentals(s) for s in buys}).T
    shortlist = scan.loc[buys, ["Company", "Close", "Stop-loss", "Qty", "Strategy works?", "Sharpe"]].join(f)
    shortlist["⭐ Best"] = ((shortlist["Strategy works?"] == "✅") & (shortlist["Profitable?"] == "✅")).map({True: "⭐", False: ""})
    display(shortlist) if "display" in globals() else print(shortlist.to_string())
else:
    print("No BUY signals today.")

## 9. Save results + paper-trading log
Set `SAVE_TO_DRIVE = True` in step 2 to keep a history across days.

In [ ]:
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/algo_scanner"
else:
    OUT = "scanner_output"
os.makedirs(OUT, exist_ok=True)

day = latest.date().isoformat()
scan.to_csv(f"{OUT}/scan_{day}.csv")

# Paper-trading log: every BUY/SELL signal, appended once per day
log_path = f"{OUT}/signal_log.csv"
new = scan[scan["Action"].isin(["🟢 BUY", "🔴 SELL"])].reset_index()[
    ["Symbol", "Company", "Action", "Close", "Stop-loss", "Qty", "Strategy works?"]]
new.insert(0, "Signal date", day)
if os.path.exists(log_path):
    old = pd.read_csv(log_path)
    old = old[old["Signal date"] != day]          # re-running same day won't duplicate
    new = pd.concat([old, new], ignore_index=True)
new.to_csv(log_path, index=False)
print(f"Saved scan_{day}.csv and updated signal_log.csv ({len(new)} signals total) in {OUT}")

## 10. Full table (all stocks)

In [ ]:
scan